In [1]:
from __future__ import annotations

from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PointStruct,
    VectorParams,
)
from sentence_transformers import SentenceTransformer

TOY_COLLECTION = "toy_cities"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QDRANT_PATH = Path("data/qdrant_local")

__all__ = [
    "TOY_COLLECTION",
    "EMBEDDING_MODEL",
    "QDRANT_PATH",
    "Distance",
    "FieldCondition",
    "Filter",
    "MatchValue",
    "PointStruct",
    "QdrantClient",
    "SentenceTransformer",
    "VectorParams",
]

client = QdrantClient(url="http://localhost:6333")
model = SentenceTransformer(EMBEDDING_MODEL)

if not client.collection_exists(collection_name=TOY_COLLECTION):
    client.create_collection(
        collection_name=TOY_COLLECTION,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )
else:
    print("toy collection already exists")

c:\Users\User\OneDrive - National University of Singapore\Latest personal projects\ClauseLens\.conda-clauselens\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3501.91it/s]


toy collection already exists


# Toy Qdrant test with cities

In [2]:
sample_texts = [
    "Berlin",
    "London",
    "Moscow",
    "New York",
    "Beijing",
    "Mumbai",
]

embeddings = model.encode(sample_texts).tolist()

operation_info = client.upsert(
    collection_name=TOY_COLLECTION,
    wait=True,
    points=[
        PointStruct(id=index + 1, vector=embedding, payload={"city": city})
        for index, (city, embedding) in enumerate(zip(sample_texts, embeddings))
    ],
)

print(operation_info)

operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>


Which of our stored vectors are most similar to the query

In [3]:
query_embedding = model.encode("capital of germany").tolist()

search_result = client.query_points(
    collection_name=TOY_COLLECTION,
    query=query_embedding,
    with_payload=True,
    limit=3
).points

print(search_result)

[ScoredPoint(id=1, version=2, score=0.7078423, payload={'city': 'Berlin'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=5, version=2, score=0.5954728, payload={'city': 'Beijing'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=6, version=2, score=0.5778409, payload={'city': 'Mumbai'}, vector=None, shard_key=None, order_value=None)]


Search only within points where city is Berlin using a filter

In [4]:
search_result = client.query_points(
    collection_name=TOY_COLLECTION,
    query=query_embedding,
    query_filter=Filter(
        must=[FieldCondition(key="city", match=MatchValue(value="Berlin"))]
    ),
    with_payload=True,
    limit=3
).points

print(search_result)

[ScoredPoint(id=1, version=2, score=0.7078423, payload={'city': 'Berlin'}, vector=None, shard_key=None, order_value=None)]


In [5]:
import json

records = []
with open('../data/processed/starter_clause_evidence.jsonl', 'r', encoding='utf-8') as file:
    for line in file:
        data_object = json.loads(line.strip())
        records.append(data_object)
        print(records)

        if len(records) == 5:
            break

[{'id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement::Anti-Assignment::1', 'document_id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement', 'source_pdf': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement.pdf', 'source_txt': 'data\\cuad\\CUAD_v1\\full_contract_txt\\Part_II\\TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement.txt', 'clause_type': 'Anti-Assignment', 'answer': 'Yes', 'text': 'Notwithstanding the foregoing, upon the prior written approval of Online BVI, which approval may be withheld in its sole discretion, the Company shall be permitted to sublicense its rights hereunder to a wholly-owned Subsidiary of the Company or a majority-owned Subsidiary of Tom Holding, for the same purpose and under the same terms and conditions as the license set forth herein.'}]
[{'id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement::Anti-Assignment::1', 'document_id': 'TomOnl